## 1. Imports

In [ ]:
import json
import re
import csv
from pathlib import Path
from collections import Counter
from math import log, sqrt

import numpy as np
from nltk.stem import PorterStemmer
from spacy.lang.en.stop_words import STOP_WORDS

# seed for reproducibility
SEED = 42
np.random.seed(SEED)

## 2. Data Loading

In [2]:
# define data directory and check for required files
DATA_DIR = Path("../../data").resolve()
dev_file = DATA_DIR / "dev_labeled_ChatGPT.jsonl"
passages_file = DATA_DIR / "passages_bm25_ChatGPT_humfacts.jsonl"

In [3]:
def read_jsonl(file_path):
    """Read a JSON Lines file into a list of dictionaries."""
    records = []

    with open(file_path, "r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {file_path}, line {line_number}"
                ) from exc

    return records

In [23]:
# Load the development and passage data
dev_data = read_jsonl(dev_file)
passages_data = read_jsonl(passages_file)

# Display information about the loaded data
print(f"Loaded {len(dev_data)} development records.")
print(f"Loaded {len(passages_data)} passage records.")

Loaded 31 development records.
Loaded 5427 passage records.


In [5]:
# Display keys and example records
print("Development record keys:")
if dev_data:
    print(list(dev_data[0].keys()))

print("\nPassage record keys:")
if passages_data:
    print(list(passages_data[0].keys()))

# Dsplay example records
if dev_data:
    print("\nExample development record:")
    print(json.dumps(dev_data[0], indent=2, ensure_ascii=False)[:2500])

# Display example passage record
if passages_data:
    print("\nExample passage record:")
    print(json.dumps(passages_data[0], indent=2, ensure_ascii=False)[:2500])

Development record keys:
['input', 'output', 'topic', 'cat', 'annotations']

Passage record keys:
['name', 'sent', 'passages']

Example development record:
{
  "input": "Question: Tell me a bio of Marianne McAndrew.",
  "output": "Marianne McAndrew is an American actress and singer, born on November 21, 1942, in Cleveland, Ohio. She began her acting career in the late 1960s, appearing in various television shows and films. Her notable film appearances include the role of Irene Molloy in the 1969 film adaptation of the musical \"Hello, Dolly!\", and as Princess Zareta in the 1976 film \"The Blue Bird\". She also had recurring roles on popular television shows such as \"The Love Boat\" and \"Fantasy Island\". In addition to her acting career, McAndrew was a trained singer and performed in various stage productions, including the Broadway production of \"The King and I\". She retired from acting in the 1990s and currently resides in California.",
  "topic": "Marianne McAndrew",
  "cat": [

## 3. Label Mapping and Example Construction

In [6]:
# Binary mapping used in the earlier results:
# S = supported
# NS = not supported
# IR = irrelevant (treated as unsupported for this experiment)

RAW_LABEL_MAP = {
    "S": 1,
    "NS": 0,
    "IR": 0,
}

fact_to_passage = {}

for record in passages_data:
    fact_text = record.get("sent")
    passages = record.get("passages") or []

    if isinstance(fact_text, str):
        fact_to_passage[fact_text] = passages

In [7]:
# store the processed examples in a list
examples = []

missing_passage_facts = 0
missing_label_facts = 0
unrecognized_label_facts = 0
observed_labels = Counter()

for record in dev_data:
    annotations = record.get("annotations") or []

    if isinstance(annotations, dict):
        annotations = [annotations]

    for sentence_annotation in annotations:
        atomic_facts = (
            sentence_annotation.get("human-atomic-facts") or []
        )

        for fact in atomic_facts:
            fact_text = fact.get("text")
            raw_label = fact.get("label")

            if not fact_text or raw_label is None:
                missing_label_facts += 1
                continue

            label = str(raw_label).strip().upper()
            observed_labels[label] += 1

            if label not in RAW_LABEL_MAP:
                unrecognized_label_facts += 1
                continue

            if fact_text not in fact_to_passage:
                missing_passage_facts += 1
                continue

            passages = fact_to_passage[fact_text] or []

            examples.append({
                "fact": fact_text,
                "passages": passages,
                "label": label,
                "gold_binary": RAW_LABEL_MAP[label],
            })

In [8]:
print("Labels found:", sorted(observed_labels.keys()))
print("Label counts:", dict(observed_labels))
print(f"Usable examples: {len(examples)}")
print(f"Missing passage matches: {missing_passage_facts}")
print(f"Missing text/label records: {missing_label_facts}")
print(f"Unrecognized labels skipped: {unrecognized_label_facts}")

if not examples:
    raise ValueError(
        "No usable examples were constructed. Inspect the JSONL schemas "
        "and confirm that fact text matches the passage records."
    )

Labels found: ['IR', 'NS', 'S']
Label counts: {'NS': 93, 'S': 112, 'IR': 16}
Usable examples: 221
Missing passage matches: 0
Missing text/label records: 0
Unrecognized labels skipped: 0


## 4. Preprocessing: Stopwords and Stemming

In [9]:
stemmer = PorterStemmer()

STOPWORDS = set("""
a an the and or but if then else when while
is am are was were be been being
to of in on at by for from with as
it its this that these those
he she they them their we our you your
i me my
""".split())

In [ ]:
stemmer = PorterStemmer()

# Keep negation words: removing them makes a fact and contradicting evidence look more similar.
NEGATION_WORDS = {"no", "not", "nor", "never", "none", "n't"}
STOPWORDS = set(STOP_WORDS) - NEGATION_WORDS

## 5. Tokenization, N-grams and Sentence Splitting

In [ ]:
def tokenize(
    text,
    lower=True,
    remove_stop=True,
    stem=True,
):
    """Clean and tokenize text with configurable preprocessing."""

    if not isinstance(text, str):
        text = ""

    if lower:
        text = text.lower()

    # Remove HTML-like tags.
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove punctuation while preserving letters, digits and whitespace.
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    tokens = text.split()

    if remove_stop:
        tokens = [
            token for token in tokens
            if token not in STOPWORDS
        ]

    if stem:
        tokens = [
            stemmer.stem(token)
            for token in tokens
        ]

    return tokens


def make_ngrams(tokens, ngram_range=(1, 2)):
    """Generate word n-grams within the requested inclusive range."""

    min_n, max_n = ngram_range
    features = []

    for n in range(min_n, max_n + 1):
        for start in range(len(tokens) - n + 1):
            features.append(
                "_".join(tokens[start:start + n])
            )

    return features


def get_passage_text(passage):
    """Support passage records stored as dictionaries or strings."""

    if isinstance(passage, str):
        return passage

    if isinstance(passage, dict):
        return str(passage.get("text", "") or "")

    return ""


def split_into_sentences(text):
    """Split passage text using punctuation followed by whitespace."""

    if not isinstance(text, str) or not text.strip():
        return []

    text = re.sub(r"<[^>]+>", " ", text)
    parts = re.split(r"(?<=[.!?])\s+", text.strip())

    return [
        part.strip()
        for part in parts
        if part.strip()
    ]


In [20]:
print("Example tokens:", tokenize("The cats were running quickly."))
print("Example unigrams and bigrams:",
      make_ngrams(["solar", "panel", "fault"], (1, 2)))

Example tokens: ['cat', 'run', 'quickli']
Example unigrams and bigrams: ['solar', 'panel', 'fault', 'solar_panel', 'panel_fault']


## 6. TF-IDF Vectors and Cosine Similarity

In [28]:
def term_frequency(features):
    """Count occurrences of each token or n-gram."""
    return Counter(features)

In [ ]:
def build_tfidf_vectors(feature_documents):
    """Build normalized TF-IDF vectors using only Python dictionaries."""

    counters = [
        term_frequency(document)
        for document in feature_documents
    ]

    document_frequency = Counter()

    for counts in counters:
        document_frequency.update(counts.keys())

    n_documents = len(counters)

    if n_documents == 0:
        return []

    idf = {
        term: log(
            (n_documents + 1) /
            (frequency + 1)
        ) + 1.0
        for term, frequency in document_frequency.items()
    }

    normalized_vectors = []

    for counts in counters:
        vector = {}

        for term, count in counts.items():
            # Log-scaled term frequency.
            tf = 1.0 + log(count)
            vector[term] = tf * idf[term]

        # L2 normalization.
        norm = sqrt(
            sum(weight * weight for weight in vector.values())
        )

        if norm > 0:
            vector = {
                term: weight / norm
                for term, weight in vector.items()
            }

        normalized_vectors.append(vector)

    return normalized_vectors